In [1]:
pip install requests pandas


Note: you may need to restart the kernel to use updated packages.


In [23]:
pip install pandas-gbq

Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas_gbq

pandas_gbq.to_gbq(df, "velib.disponibilites", project_id="velocite-velib", if_exists="append")

100%|██████████| 1/1 [00:00<00:00, 4124.19it/s]


In [26]:
import os
os.makedirs("src", exist_ok=True)

In [27]:
%%writefile src/collecte.pyv

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [25]:
import requests
import pandas as pd

URL_INFO = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
URL_STATUS = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

verifier l'api

In [3]:
reponse = requests.get(URL_INFO, timeout=30)
print("Code HTTP :", reponse.status_code)

if reponse.status_code == 200:
    stations = reponse.json()["data"]["stations"]
    print(len(stations), "stations récupérées")
else:
    print("Erreur :", reponse.status_code)

Code HTTP : 200
1519 stations récupérées


In [4]:
df_info = pd.DataFrame(stations)
df_info.head()

,station_id,stationCode,name,lat,lon,capacity,station_opening_hours,rental_methods
0,213688169,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,None,NaN
1,19179944124,40001,Hôpital Mondor,48.798922,2.453745,28,None,NaN
2,18795078746,32304,Charcot - Benfleet,48.878370,2.440524,28,None,NaN
3,36255,9020,Toudouze - Clauzel,48.879296,2.337360,21,None,[CREDITCARD]
4,251039991,14111,Cassini - Denfert-Rochereau,48.837526,2.336035,25,None,[CREDITCARD]


In [5]:
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519 entries, 0 to 1518
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   station_id             1519 non-null   int64  
 1   stationCode            1519 non-null   object 
 2   name                   1519 non-null   object 
 3   lat                    1519 non-null   float64
 4   lon                    1519 non-null   float64
 5   capacity               1519 non-null   int64  
 6   station_opening_hours  0 non-null      object 
 7   rental_methods         894 non-null    object 
dtypes: float64(2), int64(2), object(4)
memory usage: 95.1+ KB


verification des doublons

In [6]:
df_info["station_id"].duplicated().sum()

np.int64(0)

In [7]:
reponse_status = requests.get(URL_STATUS, timeout=30)
reponse_status.status_code

200

In [8]:
df_status = pd.DataFrame(reponse_status.json()["data"]["stations"])
df_status.head()

,station_id,num_bikes_available,numBikesAvailable,num_bikes_available_types,num_docks_available,numDocksAvailable,is_installed,is_returning,is_renting,last_reported,stationCode,station_opening_hours
0,213688169,3,3,"[{'mechanical': 0}, {'ebike': 3}]",30,30,1,1,1,1791808862,16107,None
1,19179944124,8,8,"[{'mechanical': 1}, {'ebike': 7}]",20,20,1,1,1,1791808752,40001,None
2,18795078746,4,4,"[{'mechanical': 3}, {'ebike': 1}]",25,25,1,1,1,1791808698,32304,None
3,36255,3,3,"[{'mechanical': 1}, {'ebike': 2}]",18,18,1,1,1,1791808948,9020,None
4,251039991,10,10,"[{'mechanical': 0}, {'ebike': 10}]",14,14,1,1,1,1791808903,14111,None


In [9]:
df_status.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1519 entries, 0 to 1518
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   station_id                 1519 non-null   int64 
 1   num_bikes_available        1519 non-null   int64 
 2   numBikesAvailable          1519 non-null   int64 
 3   num_bikes_available_types  1519 non-null   object
 4   num_docks_available        1519 non-null   int64 
 5   numDocksAvailable          1519 non-null   int64 
 6   is_installed               1519 non-null   int64 
 7   is_returning               1519 non-null   int64 
 8   is_renting                 1519 non-null   int64 
 9   last_reported              1519 non-null   int64 
 10  stationCode                1519 non-null   object
 11  station_opening_hours      0 non-null      object
dtypes: int64(9), object(3)
memory usage: 142.5+ KB


In [10]:
df_status = df_status.drop(columns=["numBikesAvailable", "numDocksAvailable", "station_opening_hours", "stationCode"])

In [11]:
df_status["nb_mecaniques"] = df_status["num_bikes_available_types"].str[0].str["mechanical"]
df_status["nb_electriques"] = df_status["num_bikes_available_types"].str[1].str["ebike"]
df_status = df_status.drop(columns=["num_bikes_available_types"])

In [12]:
df_status["last_reported"] = pd.to_datetime(df_status["last_reported"], unit="s", utc=True).dt.tz_convert("Europe/Paris")

In [13]:
df_status.head()

,station_id,num_bikes_available,num_docks_available,is_installed,is_returning,is_renting,last_reported,nb_mecaniques,nb_electriques
0,213688169,3,30,1,1,1,2026-10-12 14:41:02+02:00,0,3
1,19179944124,8,20,1,1,1,2026-10-12 14:39:12+02:00,1,7
2,18795078746,4,25,1,1,1,2026-10-12 14:38:18+02:00,3,1
3,36255,3,18,1,1,1,2026-10-12 14:42:28+02:00,1,2
4,251039991,10,14,1,1,1,2026-10-12 14:41:43+02:00,0,10


In [14]:
df_info["accepte_cb"] = df_info["rental_methods"].notna()
df_info = df_info.drop(columns=["station_opening_hours", "rental_methods"])

In [15]:
df = df_status.merge(df_info, on="station_id", how="left")

In [16]:
df.shape

(1519, 15)

In [19]:
df.head()

,station_id,num_bikes_available,num_docks_available,is_installed,is_returning,is_renting,last_reported,nb_mecaniques,nb_electriques,stationCode,name,lat,lon,capacity,accepte_cb
0,213688169,3,30,1,1,1,2026-10-12 14:41:02+02:00,0,3,16107,Benjamin Godard - Victor Hugo,48.865983,2.275725,35,False
1,19179944124,8,20,1,1,1,2026-10-12 14:39:12+02:00,1,7,40001,Hôpital Mondor,48.798922,2.453745,28,False
2,18795078746,4,25,1,1,1,2026-10-12 14:38:18+02:00,3,1,32304,Charcot - Benfleet,48.878370,2.440524,28,False
3,36255,3,18,1,1,1,2026-10-12 14:42:28+02:00,1,2,9020,Toudouze - Clauzel,48.879296,2.337360,21,True
4,251039991,10,14,1,1,1,2026-10-12 14:41:43+02:00,0,10,14111,Cassini - Denfert-Rochereau,48.837526,2.336035,25,True


Les indicateurs

In [18]:
df = df[(df["capacity"] > 0) & (df["is_installed"] == 1)]
df.shape

(1517, 15)

In [20]:
df["date_collecte"] = pd.Timestamp.now(tz="Europe/Paris")

In [21]:
df["taux_remplissage"] = df["num_bikes_available"] / df["capacity"]
df["station_vide"] = df["num_bikes_available"] == 0
df["station_pleine"] = df["num_docks_available"] == 0

In [22]:
df["station_vide"].sum(), df["station_pleine"].sum()

(np.int64(125), np.int64(76))